In [1]:
#!pip install pgmpy

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete.CPD import TabularCPD
from pgmpy.inference import VariableElimination


c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Model

### Structure

In [2]:
model = DiscreteBayesianNetwork([('ExamLevel', 'Marks'),
                         ('IQLevel', 'Marks'),
                         ('IQLevel', 'Aptiscore'),
                         ('Marks', 'Admission')])

### Relationships (the CPDs) and the probability table

In [3]:
ExamLevel = TabularCPD(
                variable = 'ExamLevel',
                variable_card = 2,
                values = [[0.7], [0.3]])

IQLevel = TabularCPD(
                variable = 'IQLevel',
                variable_card = 2,
                values = [[0.8], [0.2]])

Aptiscore = TabularCPD(
                variable = 'Aptiscore',
                variable_card = 2,
                values = [[0.75, 0.4],
                          [0.25, 0.6]],
                evidence = ['IQLevel'],
                evidence_card = [2])

Marks = TabularCPD(
                variable = 'Marks',
                variable_card = 2,
                values = [[0.6, 0.9, 0.5, 0.8],
                          [0.4, 0.1, 0.5, 0.2]],
                evidence = ['ExamLevel', 'IQLevel'],
                evidence_card = [2, 2])
				
Admission = TabularCPD(
                variable = 'Admission',
                variable_card = 2,
                values = [[0.6, 0.9],
                          [0.4, 0.1]],
                evidence = ['Marks'],
                evidence_card = [2])

### Adding relationships to models

In [4]:
model.add_cpds(ExamLevel, Marks, IQLevel, Aptiscore, Admission)

### Examining the structure of graph

In [5]:
model.get_cpds()

[<TabularCPD representing P(ExamLevel:2) at 0x27b6fa808c0>,
 <TabularCPD representing P(Marks:2 | ExamLevel:2, IQLevel:2) at 0x27b4478acf0>,
 <TabularCPD representing P(IQLevel:2) at 0x27b447884a0>,
 <TabularCPD representing P(Aptiscore:2 | IQLevel:2) at 0x27b44788f50>,
 <TabularCPD representing P(Admission:2 | Marks:2) at 0x27b44789310>]

### Find local independencies

In [6]:
model.get_independencies()

(ExamLevel ⟂ Aptiscore, IQLevel)
(ExamLevel ⟂ Admission | Marks)
(ExamLevel ⟂ IQLevel | Aptiscore)
(ExamLevel ⟂ Aptiscore | IQLevel)
(ExamLevel ⟂ Admission | Aptiscore, Marks)
(ExamLevel ⟂ Aptiscore, Admission | Marks, IQLevel)
(ExamLevel ⟂ Aptiscore | Admission, IQLevel)
(ExamLevel ⟂ Admission | Aptiscore, Marks, IQLevel)
(ExamLevel ⟂ Aptiscore | Marks, Admission, IQLevel)
(Admission ⟂ ExamLevel, Aptiscore, IQLevel | Marks)
(Admission ⟂ Aptiscore | IQLevel)
(Admission ⟂ Aptiscore, IQLevel | ExamLevel, Marks)
(Admission ⟂ Aptiscore | ExamLevel, IQLevel)
(Admission ⟂ ExamLevel, IQLevel | Aptiscore, Marks)
(Admission ⟂ ExamLevel, Aptiscore | Marks, IQLevel)
(Admission ⟂ IQLevel | Aptiscore, ExamLevel, Marks)
(Admission ⟂ Aptiscore | ExamLevel, Marks, IQLevel)
(Admission ⟂ ExamLevel | Aptiscore, Marks, IQLevel)
(Marks ⟂ Aptiscore | IQLevel)
(Marks ⟂ Aptiscore | ExamLevel, IQLevel)
(Marks ⟂ Aptiscore | Admission, IQLevel)
(Marks ⟂ Aptiscore | ExamLevel, Admission, IQLevel)
(Aptiscore ⟂ Exa

### Find active trail nodes

Active trail: For any two variables A and B in a network if any change in A influences the values of B then we say that there is an active trail between A and B.

In pgmpy active_trail_nodes gives a set of nodes which are affected (i.e. correlated) by any change in the node passed in the argument.

In [7]:
model.active_trail_nodes('Marks')

{'Marks': {'Admission', 'Aptiscore', 'ExamLevel', 'IQLevel', 'Marks'}}

In [8]:
model.active_trail_nodes('Marks', observed=['ExamLevel','IQLevel'])

{'Marks': {'Admission', 'Marks'}}

In [9]:
#Getting all the active trail nodes in the network.

model.active_trail_nodes(['Marks','ExamLevel','IQLevel','Aptiscore','Admission'])

{'Marks': {'Admission', 'Aptiscore', 'ExamLevel', 'IQLevel', 'Marks'},
 'ExamLevel': {'Admission', 'ExamLevel', 'Marks'},
 'IQLevel': {'Admission', 'Aptiscore', 'IQLevel', 'Marks'},
 'Aptiscore': {'Admission', 'Aptiscore', 'IQLevel', 'Marks'},
 'Admission': {'Admission', 'Aptiscore', 'ExamLevel', 'IQLevel', 'Marks'}}

In [10]:
#Getting all the local independencies in the network.

model.local_independencies(['ExamLevel', 'IQLevel', 'Aptiscore', 'Marks', 'Admission'])

(ExamLevel ⟂ Aptiscore, IQLevel)
(IQLevel ⟂ ExamLevel)
(Aptiscore ⟂ ExamLevel, Marks, Admission | IQLevel)
(Marks ⟂ Aptiscore | ExamLevel, IQLevel)
(Admission ⟂ ExamLevel, Aptiscore, IQLevel | Marks)

### Get all independencies

In [11]:
model.get_independencies()

(ExamLevel ⟂ Aptiscore, IQLevel)
(ExamLevel ⟂ Admission | Marks)
(ExamLevel ⟂ IQLevel | Aptiscore)
(ExamLevel ⟂ Aptiscore | IQLevel)
(ExamLevel ⟂ Admission | Aptiscore, Marks)
(ExamLevel ⟂ Aptiscore, Admission | Marks, IQLevel)
(ExamLevel ⟂ Aptiscore | Admission, IQLevel)
(ExamLevel ⟂ Admission | Aptiscore, Marks, IQLevel)
(ExamLevel ⟂ Aptiscore | Marks, Admission, IQLevel)
(Admission ⟂ ExamLevel, Aptiscore, IQLevel | Marks)
(Admission ⟂ Aptiscore | IQLevel)
(Admission ⟂ Aptiscore, IQLevel | ExamLevel, Marks)
(Admission ⟂ Aptiscore | ExamLevel, IQLevel)
(Admission ⟂ ExamLevel, IQLevel | Aptiscore, Marks)
(Admission ⟂ ExamLevel, Aptiscore | Marks, IQLevel)
(Admission ⟂ IQLevel | Aptiscore, ExamLevel, Marks)
(Admission ⟂ Aptiscore | ExamLevel, Marks, IQLevel)
(Admission ⟂ ExamLevel | Aptiscore, Marks, IQLevel)
(Marks ⟂ Aptiscore | IQLevel)
(Marks ⟂ Aptiscore | ExamLevel, IQLevel)
(Marks ⟂ Aptiscore | Admission, IQLevel)
(Marks ⟂ Aptiscore | ExamLevel, Admission, IQLevel)
(Aptiscore ⟂ Exa

# Making Inferences

In [12]:
model_infer = VariableElimination(model)

The output displays various combinations of the states of two variables, where 0 represents passed, and 1 represents failed, along with their calculated probabilities. This shows the relationship between the variables.

The table's query results provide the probabilities for the following scenarios: 
(a) having Low Marks and Failing Admission, 
(b) having Low Marks and Passing Admission, 
(c) having High Marks and Failing Admission,
(d) having High Marks and Passing Admission.

In [13]:
prob_Marks = model_infer.query(variables = ['Marks', 'Admission'])
print(prob_Marks)

+----------+--------------+------------------------+
| Marks    | Admission    |   phi(Marks,Admission) |
+==========+==============+========================+
| Marks(0) | Admission(0) |                 0.3780 |
+----------+--------------+------------------------+
| Marks(0) | Admission(1) |                 0.2520 |
+----------+--------------+------------------------+
| Marks(1) | Admission(0) |                 0.3330 |
+----------+--------------+------------------------+
| Marks(1) | Admission(1) |                 0.0370 |
+----------+--------------+------------------------+


Other combinations

In [14]:
prob_IQLevel = model_infer.query(variables = ['IQLevel', 'Marks'])
print(prob_IQLevel)

+------------+----------+----------------------+
| IQLevel    | Marks    |   phi(IQLevel,Marks) |
+============+==========+======================+
| IQLevel(0) | Marks(0) |               0.4560 |
+------------+----------+----------------------+
| IQLevel(0) | Marks(1) |               0.3440 |
+------------+----------+----------------------+
| IQLevel(1) | Marks(0) |               0.1740 |
+------------+----------+----------------------+
| IQLevel(1) | Marks(1) |               0.0260 |
+------------+----------+----------------------+


In [15]:
prob_ExamLevel = model_infer.query(variables = ['ExamLevel', 'Marks'])
print(prob_ExamLevel)

+--------------+----------+------------------------+
| ExamLevel    | Marks    |   phi(ExamLevel,Marks) |
+==============+==========+========================+
| ExamLevel(0) | Marks(0) |                 0.4620 |
+--------------+----------+------------------------+
| ExamLevel(0) | Marks(1) |                 0.2380 |
+--------------+----------+------------------------+
| ExamLevel(1) | Marks(0) |                 0.1680 |
+--------------+----------+------------------------+
| ExamLevel(1) | Marks(1) |                 0.1320 |
+--------------+----------+------------------------+


In [16]:
prob_IQLevel = model_infer.query(variables = ['IQLevel', 'Aptiscore'])
print(prob_IQLevel)

+------------+--------------+--------------------------+
| IQLevel    | Aptiscore    |   phi(IQLevel,Aptiscore) |
+============+==============+==========================+
| IQLevel(0) | Aptiscore(0) |                   0.6000 |
+------------+--------------+--------------------------+
| IQLevel(0) | Aptiscore(1) |                   0.2000 |
+------------+--------------+--------------------------+
| IQLevel(1) | Aptiscore(0) |                   0.0800 |
+------------+--------------+--------------------------+
| IQLevel(1) | Aptiscore(1) |                   0.1200 |
+------------+--------------+--------------------------+


In [17]:
prob_ExamLevel = model_infer.query(variables = ['ExamLevel', 'Admission'])
print(prob_ExamLevel)

+--------------+--------------+----------------------------+
| ExamLevel    | Admission    |   phi(ExamLevel,Admission) |
+==============+==============+============================+
| ExamLevel(0) | Admission(0) |                     0.4914 |
+--------------+--------------+----------------------------+
| ExamLevel(0) | Admission(1) |                     0.2086 |
+--------------+--------------+----------------------------+
| ExamLevel(1) | Admission(0) |                     0.2196 |
+--------------+--------------+----------------------------+
| ExamLevel(1) | Admission(1) |                     0.0804 |
+--------------+--------------+----------------------------+


### We can also get conditional probability distributions that take into account what we already know

Exam level is 0 = Easy and IQ level is 1 = High
This means that the probability of not getting admitted is 0.63 (63%) and that of getting admitted is 0.37 (37%)

In [18]:
prob = model_infer.query(variables = ['Admission'], 
                         evidence = {'ExamLevel':0, 'IQLevel':1})
print(prob)

+--------------+------------------+
| Admission    |   phi(Admission) |
+==============+==================+
| Admission(0) |           0.6300 |
+--------------+------------------+
| Admission(1) |           0.3700 |
+--------------+------------------+


Marks is 0 = Good This means that the probability of getting not admitted(0) is 0.60 (60%) and that of getting admitted(1) is 0.40 (40%).

In [19]:
prob = model_infer.query(variables = ['Admission'], 
                         evidence = {'Marks':0})
print(prob)

+--------------+------------------+
| Admission    |   phi(Admission) |
+==============+==================+
| Admission(0) |           0.6000 |
+--------------+------------------+
| Admission(1) |           0.4000 |
+--------------+------------------+


On the other hand Marks is 1 = Bad This means that the probability of getting not admitted(0) is 0.90 (90%) and that of getting admitted(1) is 0.10 (10%).

In [20]:
prob = model_infer.query(variables = ['Admission'], 
                         evidence = {'Marks':1})
print(prob)

+--------------+------------------+
| Admission    |   phi(Admission) |
+==============+==================+
| Admission(0) |           0.9000 |
+--------------+------------------+
| Admission(1) |           0.1000 |
+--------------+------------------+


Other instances / combinations

In [21]:
prob = model_infer.query(variables = ['ExamLevel'],)
print(prob)

+--------------+------------------+
| ExamLevel    |   phi(ExamLevel) |
+==============+==================+
| ExamLevel(0) |           0.7000 |
+--------------+------------------+
| ExamLevel(1) |           0.3000 |
+--------------+------------------+


In [22]:
prob = model_infer.query(variables = ['IQLevel'],)
print(prob)

+------------+----------------+
| IQLevel    |   phi(IQLevel) |
+============+================+
| IQLevel(0) |         0.8000 |
+------------+----------------+
| IQLevel(1) |         0.2000 |
+------------+----------------+


In [23]:
prob = model_infer.query(variables = ['Marks'], 
                         evidence = {'IQLevel':0, 'ExamLevel':0})
print(prob)

+----------+--------------+
| Marks    |   phi(Marks) |
+==========+==============+
| Marks(0) |       0.6000 |
+----------+--------------+
| Marks(1) |       0.4000 |
+----------+--------------+


In [24]:
prob = model_infer.query(variables = ['Marks'], 
                         evidence = {'IQLevel':0, 'ExamLevel':1})
print(prob)

+----------+--------------+
| Marks    |   phi(Marks) |
+==========+==============+
| Marks(0) |       0.5000 |
+----------+--------------+
| Marks(1) |       0.5000 |
+----------+--------------+


In [25]:
prob = model_infer.query(variables = ['Marks'], 
                         evidence = {'IQLevel':1, 'ExamLevel':0})
print(prob)

+----------+--------------+
| Marks    |   phi(Marks) |
+==========+==============+
| Marks(0) |       0.9000 |
+----------+--------------+
| Marks(1) |       0.1000 |
+----------+--------------+


In [26]:
prob = model_infer.query(variables = ['Marks'], 
                         evidence = {'ExamLevel':1, 'IQLevel':1})
print(prob)

+----------+--------------+
| Marks    |   phi(Marks) |
+==========+==============+
| Marks(0) |       0.8000 |
+----------+--------------+
| Marks(1) |       0.2000 |
+----------+--------------+


In [27]:
prob = model_infer.query(variables = ['Aptiscore'], 
                         evidence = {'IQLevel':1})
print(prob)

+--------------+------------------+
| Aptiscore    |   phi(Aptiscore) |
+==============+==================+
| Aptiscore(0) |           0.4000 |
+--------------+------------------+
| Aptiscore(1) |           0.6000 |
+--------------+------------------+


In [28]:
prob = model_infer.query(variables = ['Admission'], 
                         evidence = {'IQLevel':0, 'ExamLevel':1,})
print(prob)

+--------------+------------------+
| Admission    |   phi(Admission) |
+==============+==================+
| Admission(0) |           0.7500 |
+--------------+------------------+
| Admission(1) |           0.2500 |
+--------------+------------------+


### Most Probable State of All Variables

In [29]:
model_infer.map_query()

{'Admission': 0, 'Aptiscore': 0, 'ExamLevel': 0, 'Marks': 1, 'IQLevel': 0}

model_infer.map_query():
This function is part of a probabilistic inference library. It performs Maximum A Posteriori (MAP) estimation, which finds the most probable state (or values) for each variable in a model, given the evidence. Essentially, it's trying to determine the state of the system where all variables are the most likely to occur simultaneously.

The output presents the most probable state for each variable.
'ExamLevel': 0 The most probable level of exams is 0.
'Marks': 1  The most probable value for marks is 1.
'IQLevel': 0 The most probable IQ level is 0.
'Admission': 0 Admission is not likely (0).
'Aptiscore': 0 The most probable aptitude score is 0.

### Most probable state for a variable

In [30]:
model_infer.map_query(variables = ['Admission'])

Eliminating: IQLevel: 100%|██████████| 3/3 [00:00<00:00, 999.36it/s]


{'Admission': 0}

In [31]:
model_infer.map_query(variables = ['Marks'])

Eliminating: IQLevel: 100%|██████████| 2/2 [00:00<00:00, 666.40it/s]


{'Marks': 0}

In [32]:
model_infer.map_query(variables = ['ExamLevel'])

Finding Elimination Order: : : 0it [00:00, ?it/s]
0it [00:00, ?it/s]


{'ExamLevel': 0}

In [33]:
model_infer.map_query(variables = ['IQLevel'])

Finding Elimination Order: : : 0it [00:00, ?it/s]
0it [00:00, ?it/s]


{'IQLevel': 0}

In [34]:
model_infer.map_query(variables = ['Aptiscore'])

Eliminating: IQLevel: 100%|██████████| 1/1 [00:00<00:00, 999.83it/s]


{'Aptiscore': 0}